# Частина 1. Аналіз індексу здоров'я рослинності (VHI)

**VHI (Vegetation Health Index)** - комбінований індекс, що характеризує стан рослинності за даними супутникового зондування NOAA.  
Значення VHI лежать у діапазоні 0–100:  
- 0–10 → сильна посуха / дуже пошкоджена рослинність  
- 40–60 → задовільний стан  
- 60–100 → хороший / чудовий стан  

Дані охоплюють 27 областей України з 1981 по 2024 рік (тижневі виміри).

In [ ]:
import urllib.request
import os
import pandas as pd
from datetime import datetime

## 1. Маппінг областей

NOAA використовує власну нумерацію провінцій (`province_id` 1–27), яка не збігається з офіційними українськими кодами.  
`PROVINCE_MAP` - словник для перетворення: ключ - NOAA ID, значення - назва та офіційний UA-код.  
Також тут ініціалізується папка `data/` для збереження завантажених CSV.

In [ ]:
PROVINCE_MAP = {
    1:  {"name_ua": "Черкаська",       "new_id": 24},
    2:  {"name_ua": "Чернівецька",     "new_id": 25},
    3:  {"name_ua": "Чернігівська",    "new_id": 26},
    4:  {"name_ua": "Кримська",        "new_id": 12},
    5:  {"name_ua": "Дніпропетровська","new_id": 4},
    6:  {"name_ua": "Донецька",        "new_id": 5},
    7:  {"name_ua": "Івано-Франківська","new_id": 7},
    8:  {"name_ua": "Харківська",      "new_id": 22},
    9:  {"name_ua": "Херсонська",      "new_id": 23},
    10: {"name_ua": "Хмельницька",     "new_id": 24},
    11: {"name_ua": "Київська",        "new_id": 10},
    12: {"name_ua": "Кіровоградська",  "new_id": 11},
    13: {"name_ua": "Луганська",       "new_id": 13},
    14: {"name_ua": "Львівська",       "new_id": 14},
    15: {"name_ua": "Миколаївська",    "new_id": 15},
    16: {"name_ua": "Одеська",         "new_id": 16},
    17: {"name_ua": "Полтавська",      "new_id": 17},
    18: {"name_ua": "Рівненська",      "new_id": 18},
    19: {"name_ua": "Сумська",         "new_id": 19},
    20: {"name_ua": "Тернопільська",   "new_id": 20},
    21: {"name_ua": "Закарпатська",    "new_id": 6},
    22: {"name_ua": "Вінницька",       "new_id": 1},
    23: {"name_ua": "Волинська",       "new_id": 2},
    24: {"name_ua": "Запорізька",      "new_id": 8},
    25: {"name_ua": "Житомирська",     "new_id": 9},
    26: {"name_ua": "Київ (місто)",    "new_id": 10},
    27: {"name_ua": "Севастополь",     "new_id": 21},
}

DATA_DIR = "data"
os.makedirs(DATA_DIR, exist_ok=True)

## 2. Завантаження VHI-даних

Функція `download_vhi` виконує такі кроки:
1. Перевіряє, чи файл уже існує (щоб не завантажувати повторно).
2. Формує URL до API NOAA для конкретної провінції та діапазону 1981–2024.
3. Зберігає CSV-файл з міткою часу у назві.

Нижче цикл, що завантажує дані для всіх 27 провінцій.

In [ ]:
def download_vhi(province_id: int) -> str:
    """
    Завантажує VHI-файл для вказаної області.
    Повертає шлях до збереженого файлу або None, якщо файл вже існує.
    Механізм: перевіряємо чи є вже файл для даного province_id.
    """
    existing = [f for f in os.listdir(DATA_DIR) if f.startswith(f"vhi_{province_id}_")]
    if existing:
        print(f"[SKIP] Province {province_id}: файл вже існує → {existing[0]}")
        return os.path.join(DATA_DIR, existing[0])
    
    url = (
        f"https://www.star.nesdis.noaa.gov/smcd/emb/vci/VH/get_TS_admin.php"
        f"?country=UKR&provinceID={province_id}&year1=1981&year2=2024&type=Mean"
    )
    timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")
    filename = f"vhi_{province_id}_{timestamp}.csv"
    filepath = os.path.join(DATA_DIR, filename)
    
    try:
        urllib.request.urlretrieve(url, filepath)
        print(f"[OK] Province {province_id} → {filename}")
        return filepath
    except Exception as e:
        print(f"[ERROR] Province {province_id}: {e}")
        return None

# Завантажуємо всі 27 областей (окрім 0 - середнє по Україні)
for pid in range(1, 28):
    download_vhi(pid)

## 3. Завантаження та об'єднання файлів у DataFrame

Функція `load_vhi_files` зчитує всі CSV-файли з папки `data/` та виконує:
- **Видалення сміттєвих рядків** - рядки з `<br>` або нечисловим роком відкидаються.
- **Конвертація типів** - колонки `Year`, `Week`, `VHI` та ін. приводяться до числових.
- **Заповнення пропусків** - NaN замінюються медіаною відповідного стовпця (не спотворює розподіл).
- **Додавання ідентифікаторів** - до кожного рядка додається NOAA-код, UA-код та назва області.

Результат - єдиний `df_vhi` із даними всіх областей.

In [ ]:
def load_vhi_files() -> pd.DataFrame:
    """
    Читає всі VHI CSV-файли з папки data/, очищає та об'єднує в один DataFrame.
    """
    all_frames = []
    
    for filename in sorted(os.listdir(DATA_DIR)):
        if not filename.endswith(".csv"):
            continue
        
        parts = filename.split("_")
        province_id = int(parts[1])
        
        filepath = os.path.join(DATA_DIR, filename)
        
        df = pd.read_csv(
            filepath,
            header=1,
            skipinitialspace=True
        )
        
        df.columns = df.columns.str.strip()
        
        df = df[df['Year'].apply(lambda x: str(x).strip().lstrip('<').isdigit())]
        
        df['Year'] = df['Year'].astype(int)
        df['Week'] = df['Week'].astype(int)
        df['SMN'] = pd.to_numeric(df['SMN'], errors='coerce')
        df['SMT'] = pd.to_numeric(df['SMT'], errors='coerce')
        df['VCI'] = pd.to_numeric(df['VCI'], errors='coerce')
        df['TCI'] = pd.to_numeric(df['TCI'], errors='coerce')
        df['VHI'] = pd.to_numeric(df['VHI'], errors='coerce')
        
        for col in ['SMN', 'SMT', 'VCI', 'TCI', 'VHI']:
            df[col].fillna(df[col].median(), inplace=True)
        
        df['province_id_noaa'] = province_id
        df['province_id_ua']   = PROVINCE_MAP[province_id]['new_id']
        df['province_name']    = PROVINCE_MAP[province_id]['name_ua']
        
        all_frames.append(df)
    
    result = pd.concat(all_frames, ignore_index=True)
    return result

df_vhi = load_vhi_files()
print(f"Загальна кількість рядків: {len(df_vhi)}")
print(df_vhi.head(10))
print(df_vhi.info())

## 4. Перевірка маппінгу індексів

Виводимо таблицю відповідності між NOAA-кодом і офіційним UA-кодом, щоб переконатись у коректності перетворення.

In [ ]:
# Вже зроблено під час завантаження: стовпець province_id_ua
# Перевіримо відповідність
print("Перевірка маппінгу індексів:")
print(df_vhi[['province_id_noaa', 'province_id_ua', 'province_name']].drop_duplicates().sort_values('province_id_ua'))

## 5. VHI для конкретної області та року

Функція `vhi_for_region_year` повертає тижневий ряд VHI для однієї області в одному конкретному році.  
Параметр `province_id_ua` - офіційний UA-код (після маппінгу).  

**Приклад:** Вінницька область (UA-код = 1) за 2020 рік.

In [ ]:
def vhi_for_region_year(df: pd.DataFrame, province_id_ua: int, year: int) -> pd.DataFrame:
    """
    Повертає ряд VHI для вказаної області (за укр. індексом) та року.
    """
    result = df[(df['province_id_ua'] == province_id_ua) & (df['Year'] == year)][
        ['Week', 'VHI', 'province_name', 'Year']
    ]
    return result.reset_index(drop=True)

# Виклик:
result1 = vhi_for_region_year(df_vhi, province_id_ua=1, year=2020)  # Вінницька, 2020
print("VHI для Вінницької області, 2020 рік:")
print(result1)

## 6. VHI для кількох областей за діапазон років

Функція `vhi_range_regions` дозволяє одночасно фільтрувати кілька областей та задати часовий діапазон.  
Вхідний параметр `province_ids` - список UA-кодів.

**Приклад:** Вінницька, Чернівецька, Чернігівська (коди 1, 2, 3) за 2018–2020 роки.

In [ ]:
def vhi_range_regions(df: pd.DataFrame, province_ids: list, year_from: int, year_to: int) -> pd.DataFrame:
    """
    Повертає VHI за діапазон років для вказаних областей.
    """
    result = df[
        (df['province_id_ua'].isin(province_ids)) &
        (df['Year'] >= year_from) &
        (df['Year'] <= year_to)
    ][['Year', 'Week', 'VHI', 'province_name', 'province_id_ua']]
    return result.reset_index(drop=True)

# Виклик:
result2 = vhi_range_regions(df_vhi, province_ids=[1, 2, 3], year_from=2018, year_to=2020)
print("VHI для областей 1-3 (2018-2020):")
print(result2)

## 7. Статистика VHI по областях

Функція `vhi_stats` обчислює **мінімум, максимум, середнє та медіану** VHI для кожної із вказаних областей у заданому часовому діапазоні.

- **Min / Max** - крайні значення (виявлення посух або аномально доброго стану).
- **Mean** - середній рівень здоров'я рослинності.
- **Median** - робустна оцінка центру, стійка до викидів.

**Приклад:** Вінницька (1), Чернівецька (2), Кримська (4) за 2015–2024 роки.

In [ ]:
def vhi_stats(df: pd.DataFrame, province_ids: list, year_from: int, year_to: int) -> pd.DataFrame:
    """
    Повертає min, max, середнє, медіану VHI для вказаних областей та років.
    """
    filtered = df[
        (df['province_id_ua'].isin(province_ids)) &
        (df['Year'] >= year_from) &
        (df['Year'] <= year_to)
    ]
    stats = filtered.groupby(['province_name', 'province_id_ua'])['VHI'].agg(
        Min='min', Max='max', Mean='mean', Median='median'
    ).reset_index()
    return stats

# Виклик:
result3 = vhi_stats(df_vhi, province_ids=[1, 2, 4], year_from=2015, year_to=2024)
print("Статистика VHI:")
print(result3)